# World Bank Indicators API Example

> The following is an example of a [Jupyter notebook](https://jupyter.org) - a tutorial of how to retrieve data from the [World Bank Indicators API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation) - that illustrates how to use computational content with the [template](https://worldbank.github.io/template). 

print("hello")

In [2]:
import pandas as pd

# Load the Piece 1 output file
FILE = "open_router_data_classified.csv"   # change if needed
df = pd.read_csv(FILE)

print("=== Basic shape ===")
print(df.shape)

print("\n=== Columns ===")
print(df.columns.tolist())

# --------------------------------------------------
# 1. Overall classification counts
# --------------------------------------------------
print("\n=== classification_source counts ===")
print(df["classification_source"].value_counts(dropna=False))

# --------------------------------------------------
# 2. Flag counts
# --------------------------------------------------
flag_cols = [
    "is_open_source",
    "is_chinese_model",
    "is_unknown_country",
    "is_unclassified_model",
    "is_missing_model_slug",
]

print("\n=== Flag counts ===")
for col in flag_cols:
    if col in df.columns:
        print(f"{col}: {df[col].fillna(False).sum()}")

# --------------------------------------------------
# 3. Rows and token share by classification status
# --------------------------------------------------
summary = pd.DataFrame({
    "rows": df["classification_source"].value_counts(dropna=False),
    "tokens": df.groupby("classification_source", dropna=False)["total_tokens"].sum()
}).fillna(0)

summary["row_pct"] = (summary["rows"] / len(df) * 100).round(2)
summary["token_pct"] = (summary["tokens"] / df["total_tokens"].sum() * 100).round(2)

print("\n=== Rows and token share by classification_source ===")
print(summary.sort_values("tokens", ascending=False))

# --------------------------------------------------
# 4. Unique model counts
# --------------------------------------------------
model_summary = (
    df.groupby("classification_source", dropna=False)["model_permaslug"]
      .nunique()
      .rename("unique_models")
      .reset_index()
      .sort_values("unique_models", ascending=False)
)

print("\n=== Unique model slugs by classification_source ===")
print(model_summary)

# --------------------------------------------------
# 5. Top unclassified models by rows and tokens
# --------------------------------------------------
unclassified = df[df["classification_source"] == "unclassified"].copy()

if len(unclassified) == 0:
    print("\n=== No unclassified models found ===")
else:
    top_unclassified = (
        unclassified.groupby("model_permaslug", dropna=False)
        .agg(
            rows=("model_permaslug", "size"),
            total_tokens=("total_tokens", "sum"),
            countries=("country", "nunique"),
            months=("month", "nunique"),
        )
        .reset_index()
        .sort_values(["total_tokens", "rows"], ascending=[False, False])
    )

    print("\n=== Top unclassified models ===")
    print(top_unclassified.head(50))

# --------------------------------------------------
# 6. Top missing / unknown data issues
# --------------------------------------------------
print("\n=== Unknown country rows ===")
unknown_country = df[df["is_unknown_country"] == True]
print(len(unknown_country))

print("\n=== Missing model slug rows ===")
missing_slug = df[df["is_missing_model_slug"] == True]
print(len(missing_slug))

# --------------------------------------------------
# 7. Quick monthly view of classified vs unclassified
# --------------------------------------------------
monthly_status = (
    df.assign(
        status=df["classification_source"].where(
            df["classification_source"].isin(["unclassified", "missing_slug"]),
            "classified"
        )
    )
    .groupby(["month", "status"], dropna=False)
    .agg(
        rows=("status", "size"),
        total_tokens=("total_tokens", "sum")
    )
    .reset_index()
    .sort_values(["month", "status"])
)

print("\n=== Monthly classified vs unclassified summary ===")
print(monthly_status.head(100))

# --------------------------------------------------
# 8. Optional: save review tables
# --------------------------------------------------
summary.to_csv("classification_summary.csv")
model_summary.to_csv("classification_unique_models.csv", index=False)
monthly_status.to_csv("classification_monthly_status.csv", index=False)

if len(unclassified) > 0:
    top_unclassified.to_csv("top_unclassified_models.csv", index=False)

print("\nSaved review files:")
print("- classification_summary.csv")
print("- classification_unique_models.csv")
print("- classification_monthly_status.csv")
if len(unclassified) > 0:
    print("- top_unclassified_models.csv")

=== Basic shape ===
(385932, 11)

=== Columns ===
['month', 'model_permaslug', 'provider', 'country', 'total_tokens', 'is_open_source', 'is_chinese_model', 'classification_source', 'is_unknown_country', 'is_unclassified_model', 'is_missing_model_slug']

=== classification_source counts ===
classification_source
provider_rule         300851
unclassified           70014
exact_override         10335
heuristic_deepseek      4034
heuristic_qwen           698
Name: count, dtype: int64

=== Flag counts ===
is_open_source: 151008
is_chinese_model: 94618
is_unknown_country: 4019
is_unclassified_model: 70014
is_missing_model_slug: 0

=== Rows and token share by classification_source ===
                         rows           tokens  row_pct  token_pct
classification_source                                             
provider_rule          300851  223339443831109    77.95      94.25
unclassified            70014    5988162770733    18.14       2.53
exact_override          10335    4910766403257